# Lesson 0 · pandas crash course: the 10 operations you will use most

Each operation is shown first on a **6-row toy table**, so the syntax is easy to see, and then used straight away on the real FlyWire annotation table `ann`.

Every section follows the same pattern: **explanation → worked example (already written, just run it) → TODO cell (you write it)**.

**Two core objects**
- **DataFrame**: a whole table with rows and columns, like one sheet in Excel.
- **Series**: a single column of a table, i.e. the one-dimensional version of a DataFrame.

> You can do this lesson first, or come back to it whenever a later lesson uses a pandas operation you have not seen.

## 0. Setup: a toy table and the FlyWire table

In [1]:
import pandas as pd

toy = pd.DataFrame({
    "name":       ["PN_a", "PN_b", "KC_1", "KC_2", "KC_3", "APL"],
    "cell_class": ["ALPN", "ALPN", "Kenyon_Cell", "Kenyon_Cell", "Kenyon_Cell", "APL"],
    "nt":         ["acetylcholine", "gaba", "acetylcholine", "acetylcholine", "acetylcholine", "gaba"],
    "side":       ["right", "left", "right", "right", "left", "right"],
    "n_syn":      [120, 80, 15, 22, 9, 3000],
})
ann = pd.read_csv("../data/neuron_annotations.tsv", sep="\t", low_memory=False)
toy

,name,cell_class,nt,side,n_syn
0,PN_a,ALPN,acetylcholine,right,120
1,PN_b,ALPN,gaba,left,80
2,KC_1,Kenyon_Cell,acetylcholine,right,15
3,KC_2,Kenyon_Cell,acetylcholine,right,22
4,KC_3,Kenyon_Cell,acetylcholine,left,9
5,APL,APL,gaba,right,3000


## 1. Look at the table: `.head()`, `.shape`, `.columns`

- `df.head(n)`: the first n rows (5 by default);
- `df.shape`: (number of rows, number of columns). Note there are no parentheses: it is an **attribute**, not a function;
- `df.columns`: all column names.

In [2]:
print(toy.shape)
print(toy.columns.tolist())
toy.head(3)

(6, 5)
['name', 'cell_class', 'nt', 'side', 'n_syn']


,name,cell_class,nt,side,n_syn
0,PN_a,ALPN,acetylcholine,right,120
1,PN_b,ALPN,gaba,left,80
2,KC_1,Kenyon_Cell,acetylcholine,right,15


**TODO 1**: print the shape of `ann` and show its first 3 rows.

In [3]:
# TODO 1
print(ann.shape)
ann.head(3)

(139248, 31)


,supervoxel_id,root_id,pos_x,pos_y,pos_z,soma_x,soma_y,soma_z,nucleus_id,flow,...,known_nt_source,side,nerve,vfb_id,fbbt_id,status,dimorphism,matching_notes,fru_dsx,synonyms
0,78112261444987077,720575940628857210,109306.0,50491.0,3960.0,104904.0,47464.0,5461.0,2453924.0,intrinsic,...,NaN,left,NaN,fw138205,FBbt_20001935,NaN,isomorphic,NaN,NaN,NaN
1,82475466912542440,720575940626838909,172029.0,55635.0,1592.0,177472.0,56936.0,1429.0,7393349.0,intrinsic,...,NaN,right,NaN,fw000001,NaN,NaN,isomorphic,NaN,NaN,NaN
2,83038623024880664,720575940626046919,180632.0,58664.0,1925.0,180632.0,58664.0,1925.0,7415038.0,intrinsic,...,NaN,right,NaN,fw000002,FBbt_20000538,NaN,isomorphic,NaN,NaN,NaN


## 2. Select columns: one pair of brackets `[]` for one column, two `[[ ]]` for several

- `toy["nt"]` returns a **Series** (one column);
- `toy[["name", "nt"]]` returns a **DataFrame** (several columns). The inner `[]` is an ordinary Python list.
- If a column name has no spaces you can also write `toy.nt`, which is the same as `toy["nt"]`.

In [4]:
print(toy["nt"])
toy[["name", "nt"]]

0    acetylcholine
1             gaba
2    acetylcholine
3    acetylcholine
4    acetylcholine
5             gaba
Name: nt, dtype: str


,name,nt
0,PN_a,acetylcholine
1,PN_b,gaba
2,KC_1,acetylcholine
3,KC_2,acetylcholine
4,KC_3,acetylcholine
5,APL,gaba


**TODO 2**: select the three columns `cell_type`, `top_nt` and `side` from `ann` and show the first 5 rows.

In [5]:
# TODO 2
ann[["cell_type", "top_nt", "side"]].head(5)

,cell_type,top_nt,side
0,PS180,acetylcholine,left
1,CB0924,acetylcholine,right
2,AVLP429,acetylcholine,right
3,AVLP151,acetylcholine,right
4,LC27,acetylcholine,right


## 3. Filter rows with a boolean mask (the single most important operation)

Think of it in two steps:
1. `toy["side"] == "right"` produces a column of True/False values, called a **mask**;
2. `toy[mask]` keeps only the rows where the mask is True.

Usually both steps are written on one line: `toy[toy.side == "right"]`.

In [6]:
toy[toy.side == "right"]

,name,cell_class,nt,side,n_syn
0,PN_a,ALPN,acetylcholine,right,120
2,KC_1,Kenyon_Cell,acetylcholine,right,15
3,KC_2,Kenyon_Cell,acetylcholine,right,22
5,APL,APL,gaba,right,3000


**TODO 3**: keep the rows of `ann` with `cell_class == "MBON"`, store them as `mbons`, and print the number of rows.

In [7]:
# TODO 3
mbons = ann[ann.cell_class == "MBON"].copy()
print(len(mbons))

96


## 4. Several conditions: `&` (and), `|` (or), `~` (not). Wrap every condition in parentheses

```python
toy[(toy.side == "right") & (toy.nt == "gaba")]
```

Without the parentheses you get an error, because `&` binds more tightly than `==`.

**`.isin([...])`** tests whether a value equals any item in a list. It is shorter than chaining many `|`.

In [8]:
print(toy[(toy.side == "right") & (toy.nt == "acetylcholine")])
toy[toy.cell_class.isin(["ALPN", "APL"])]

   name   cell_class             nt   side  n_syn
0  PN_a         ALPN  acetylcholine  right    120
2  KC_1  Kenyon_Cell  acetylcholine  right     15
3  KC_2  Kenyon_Cell  acetylcholine  right     22


,name,cell_class,nt,side,n_syn
0,PN_a,ALPN,acetylcholine,right,120
1,PN_b,ALPN,gaba,left,80
5,APL,APL,gaba,right,3000


**TODO 4**: from `ann`, keep the rows on the **right side** whose `cell_class` is `"MBON"` or `"DAN"`, and print the number of rows.

In [9]:
# TODO 4
print(len(ann[(ann.side == "right") & ann.cell_class.isin(["MBON", "DAN"])]))

213


## 5. Summarise a column: `.value_counts()`, `.nunique()`, `.describe()`

- `.value_counts()`: how many times each value appears, sorted from most to least common;
- `.nunique()`: how many distinct values there are;
- `.describe()`: a summary of a numeric column: count, mean, std, min, quartiles and max.

In [10]:
print(toy.nt.value_counts())
print(toy.cell_class.nunique())
toy.n_syn.describe()

nt
acetylcholine    4
gaba             2
Name: count, dtype: int64
3


count       6.000000
mean      541.000000
std      1205.446307
min         9.000000
25%        16.750000
50%        51.000000
75%       110.000000
max      3000.000000
Name: n_syn, dtype: float64

**TODO 5**: how many different values does `super_class` take in `ann`? How many neurons are there in each?

In [11]:
# TODO 5
print(ann.super_class.nunique())
ann.super_class.value_counts()

10


super_class
optic                 77541
central               32383
sensory               16907
visual_projection      8038
ascending              1750
descending             1303
sensory_ascending       612
visual_centrifugal      524
motor                   110
endocrine                80
Name: count, dtype: int64

## 6. Create a new column

Assigning to a new column name creates that column. The operation is applied row by row:

```python
toy["strong"] = toy.n_syn > 50
```

> If you add a column to a table you filtered out of a bigger one, pandas may show a `SettingWithCopyWarning`. The fix is to add `.copy()` when filtering: `sub = df[mask].copy()`.

In [12]:
toy["strong"] = toy.n_syn > 50
toy["n_syn_x2"] = toy.n_syn * 2
toy

,name,cell_class,nt,side,n_syn,strong,n_syn_x2
0,PN_a,ALPN,acetylcholine,right,120,True,240
1,PN_b,ALPN,gaba,left,80,True,160
2,KC_1,Kenyon_Cell,acetylcholine,right,15,False,30
3,KC_2,Kenyon_Cell,acetylcholine,right,22,False,44
4,KC_3,Kenyon_Cell,acetylcholine,left,9,False,18
5,APL,APL,gaba,right,3000,True,6000


**TODO 6**: from `ann`, keep the Kenyon cells (`cell_class == "Kenyon_Cell"`) on the right side as `kcs` (remember `.copy()`). Add a column `is_gamma` equal to `kcs.cell_type == "KCg-m"`. Then count how many are True. Hint: `.sum()` counts True values.

In [13]:
# TODO 6
kcs = ann[(ann.cell_class == "Kenyon_Cell") & (ann.side == "right")].copy()
kcs["is_gamma"] = kcs.cell_type == "KCg-m"
print(kcs.is_gamma.sum())

1120


## 7. Text columns: the `.str` methods

For a text column, add `.str` and then call a string method; pandas runs it on every row:
- `.str.startswith("KC")`: does the value start with a given string? Returns True/False;
- `.str.replace("ORN_", "")`: replace text;
- `.str.split("_").str[0]`: split on `_` and take piece number 0.

> Missing values (NaN) give NaN back from `.str` methods. Convert with `.astype(str)` first if that gets in the way.

In [14]:
print(toy.name.str.startswith("KC"))
print(toy.name.str.split("_").str[0])

0    False
1    False
2     True
3     True
4     True
5    False
Name: name, dtype: bool
0     PN
1     PN
2     KC
3     KC
4     KC
5    APL
Name: name, dtype: object


**TODO 7**: for `kcs` from TODO 6, add a column `kc_class` equal to piece 0 of `cell_type` split on `-` (e.g. `KCg-m` → `KCg`), then count the values with `.value_counts()`.

In [15]:
# TODO 7
kcs["kc_class"] = kcs.cell_type.str.split("-").str[0]
kcs.kc_class.value_counts()

kc_class
KCg       1270
KCab       860
KCapbp     466
KCa'b'       1
Name: count, dtype: int64

## 8. Group and summarise: `.groupby()`

Read it as: **group by column A, then compute some statistic of column B within each group.**

```python
toy.groupby("cell_class").n_syn.mean()        # mean synapse count per class
toy.groupby("cell_class").nt.value_counts()   # transmitter counts within each class
toy.groupby("cell_class").size()              # number of rows per group
```

In [16]:
print(toy.groupby("cell_class").n_syn.mean())
toy.groupby("cell_class").nt.value_counts()

cell_class
ALPN            100.000000
APL            3000.000000
Kenyon_Cell      15.333333
Name: n_syn, dtype: float64


cell_class   nt           
ALPN         acetylcholine    1
             gaba             1
APL          gaba             1
Kenyon_Cell  acetylcholine    3
Name: count, dtype: int64

**TODO 8**: using the rows of `ann` with `cell_class == "ALPN"`, group by `cell_sub_class` and count the `top_nt` values in each group. (This is the bonus question of Lesson 1.)

In [17]:
# TODO 8
pns = ann[ann.cell_class == "ALPN"]
pns.groupby("cell_sub_class").top_nt.value_counts()

cell_sub_class   top_nt       
multiglomerular  acetylcholine    222
                 gaba             170
                 glutamate          8
uniglomerular    acetylcholine    246
                 gaba              31
Name: count, dtype: int64

## 9. Sort: `.sort_values()`

```python
toy.sort_values("n_syn")                    # smallest first
toy.sort_values("n_syn", ascending=False)   # largest first
```

In [18]:
toy.sort_values("n_syn", ascending=False)

,name,cell_class,nt,side,n_syn,strong,n_syn_x2
5,APL,APL,gaba,right,3000,True,6000
0,PN_a,ALPN,acetylcholine,right,120,True,240
1,PN_b,ALPN,gaba,left,80,True,160
3,KC_2,Kenyon_Cell,acetylcholine,right,22,False,44
2,KC_1,Kenyon_Cell,acetylcholine,right,15,False,30
4,KC_3,Kenyon_Cell,acetylcholine,left,9,False,18


**TODO 9**: among the right-side uniglomerular PNs (`cell_class == "ALPN"` and `cell_sub_class == "uniglomerular"`), count the neurons of each `cell_type`, sort from largest to smallest and show the top 5. Hint: `groupby(...).size()` returns a Series that you can `.sort_values(ascending=False)`; or simply use `value_counts()`, which is already sorted.

In [19]:
# TODO 9
upns = ann[(ann.side == "right") & (ann.cell_class == "ALPN") & (ann.cell_sub_class == "uniglomerular")]
upns.groupby("cell_type").size().sort_values(ascending=False).head(5)
# equivalent: upns.cell_type.value_counts().head(5)

cell_type
DA1_lPN      7
DA2_lPN      6
VM5d_adPN    5
VA1v_adPN    5
DL3_lPN      5
dtype: int64

## 10. Join two tables: `.merge()`

A connection table only has a `pre` and a `post` neuron ID. To know what kind of cell each one is, you have to look it up in the annotation table. `merge` does this lookup automatically, like VLOOKUP in Excel.

```python
edges.merge(toy[["name", "cell_class"]], left_on="pre", right_on="name")
```

This means: match the `pre` column of `edges` against the `name` column of `toy`, and attach the matching row's `cell_class`.

In [20]:
edges = pd.DataFrame({"pre": ["PN_a", "PN_a", "PN_b"], "post": ["KC_1", "KC_2", "KC_3"], "syn_count": [7, 12, 5]})
edges.merge(toy[["name", "cell_class"]], left_on="pre", right_on="name")

,pre,post,syn_count,name,cell_class
0,PN_a,KC_1,7,PN_a,ALPN
1,PN_a,KC_2,12,PN_a,ALPN
2,PN_b,KC_3,5,PN_b,ALPN


**TODO 10**: merge `edges` with `toy` twice, to get the `cell_class` of the pre neuron and of the post neuron.

Hints:
- In the second merge both tables have a `cell_class` column, so pandas adds the suffixes `_x` and `_y` automatically.
- You can choose your own suffixes with `suffixes=("_pre", "_post")`.
- At the end keep only `pre`, `post`, `syn_count` and the two class columns.

In [21]:
# TODO 10
lookup = toy[["name", "cell_class"]]
both = (edges
        .merge(lookup, left_on="pre", right_on="name")
        .merge(lookup, left_on="post", right_on="name", suffixes=("_pre", "_post")))
both[["pre", "post", "syn_count", "cell_class_pre", "cell_class_post"]]

,pre,post,syn_count,cell_class_pre,cell_class_post
0,PN_a,KC_1,7,ALPN,Kenyon_Cell
1,PN_a,KC_2,12,ALPN,Kenyon_Cell
2,PN_b,KC_3,5,ALPN,Kenyon_Cell


## Cheat sheet

| What you want | How to write it |
|---|---|
| First rows | `df.head()` |
| One column / several | `df["a"]` / `df[["a", "b"]]` |
| Filter rows | `df[df.a == 1]` |
| Several conditions | `df[(cond1) & (cond2)]`; `\|` is or, `~` is not |
| Value in a list | `df[df.a.isin([...])]` |
| Counting | `df.a.value_counts()`, `df.a.nunique()`, `len(df)` |
| Summary statistics | `df.a.describe()`, `.mean()`, `.sum()` |
| New column | `df["new"] = ...` (use `.copy()` on filtered tables) |
| Text | `df.a.str.startswith()`, `.str.replace()`, `.str.split("_").str[0]` |
| Group statistics | `df.groupby("a").b.mean()` / `.size()` / `.value_counts()` |
| Sort | `df.sort_values("a", ascending=False)` |
| Lookup join | `df1.merge(df2, left_on="x", right_on="y")` |

**Two common traps**
1. Forgetting parentheses with several conditions: `df[df.a == 1 & df.b == 2]` raises an error.
2. One pair of brackets or two: `df["a"]` is a Series; `df[["a"]]` is a DataFrame with one column.